# LeNet logits from 5151 PCA-quadratic coordinates

$z=\mathrm{PCA}_{100}(X/255)$; features are the 100 scores $z_k$, the 5050 products
$z_iz_j\ (i\le j)$, and a bias — **5151** coordinates. Readout $\hat Y=[\phi\mid 1]W$
onto the 10 LeNet logits.

LeNet itself sees **raw pixels in $[0,255]$**; the PCA/quadratic features use $X/255$.

## Full experiment requirements
Install NumPy, PyTorch and ONNX. This reproduces the original algorithm with 100 PCs and all 60,000 training images. The original ONNX weights are included in models/lenet.onnx. The float64 60,000 × 5,151 design matrix alone is about 2.47 GB; least-squares workspace and intermediate arrays need additional memory. Use a high-memory runtime. Published numbers below are saved results from the supplied notebooks, not results newly executed here.


In [ ]:
%pip install numpy torch onnx

In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, shutil
# Colab downloads the public repository automatically. Locally, use a checkout.
ROOT = Path.cwd()
if not (ROOT / "approximation.py").exists():
    ROOT = Path.cwd() / "lenet-examples"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/andreymathcom/lenet-examples.git", str(ROOT)], check=True)
if not (ROOT / "data/Y_lenet_mnist.npy").exists():
    with zipfile.ZipFile(ROOT / "lenet-data.zip") as archive:
        for member in archive.infolist():
            if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError("Unsafe archive path")
        archive.extractall(ROOT)
(ROOT / "models").mkdir(exist_ok=True)
if not (ROOT / "models/lenet.onnx").exists():
    shutil.copyfile(ROOT / "lenet.onnx", ROOT / "models/lenet.onnx")
sys.path.insert(0, str(ROOT))
print("Project:", ROOT)


In [ ]:
from pathlib import Path
import gzip, struct
import numpy as np
import torch, torch.nn as nn, onnx
from onnx import numpy_helper

DATA = ROOT / "data"
LENET_ONNX = ROOT / "models/lenet.onnx"
if not LENET_ONNX.exists():
    raise FileNotFoundError("Add the original lenet.onnx to models/. It was not in the uploaded archive.")


def _images(name):
    with gzip.open(DATA / "mnist_data" / name, "rb") as f:
        _, n, r, c = struct.unpack(">IIII", f.read(16))
        return np.frombuffer(f.read(), np.uint8).reshape(n, 1, r, c).astype(np.float64)


def _labels(name):
    with gzip.open(DATA / "mnist_data" / name, "rb") as f:
        struct.unpack(">II", f.read(8))
        return np.frombuffer(f.read(), np.uint8).copy()


X_img = _images("train-images-idx3-ubyte.gz")
X_img_test = _images("t10k-images-idx3-ubyte.gz")
labels = _labels("train-labels-idx1-ubyte.gz")
labels_test = _labels("t10k-labels-idx1-ubyte.gz")
X = X_img.reshape(len(X_img), -1)
X_test = X_img_test.reshape(len(X_img_test), -1)


class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 5, padding=2)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 64, 5, padding=2)
        self.conv3 = nn.Conv2d(64, 1024, 7)
        self.conv4 = nn.Conv2d(1024, 10, 1)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        return self.conv4(torch.relu(self.conv3(x))).flatten(1)


ONNX_KEYS = {
    "conv1": "conv1first",
    "conv2": "conv2",
    "conv3": "conv3",
    "conv4": "conv4last",
}
init = {t.name: numpy_helper.to_array(t) for t in onnx.load(str(LENET_ONNX)).graph.initializer}
net, sd = LeNet(), None
sd = net.state_dict()
for torch_name, onnx_name in ONNX_KEYS.items():
    sd[f"{torch_name}.weight"] = torch.from_numpy(np.array(init[f"import/{onnx_name}/Variable:0"], np.float32))
    sd[f"{torch_name}.bias"] = torch.from_numpy(np.array(init[f"import/{onnx_name}/Variable_1:0"], np.float32))
net.load_state_dict(sd)
net.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net.to(device)


def logits(x_nchw):
    out = []
    with torch.no_grad():
        for s in range(0, len(x_nchw), 256):
            xb = torch.from_numpy(x_nchw[s : s + 256].astype(np.float32)).to(device)
            out.append(net(xb).cpu().numpy())
    return np.concatenate(out).astype(np.float64)


Y_PATH = DATA / "Y_lenet_mnist.npy"
Y = np.load(Y_PATH).astype(np.float64) if Y_PATH.exists() else logits(X_img)
Y_test = logits(X_img_test)
print("X", X.shape, "Y", Y.shape, "Y_test", Y_test.shape)
print(f"LeNet train {100 * (Y.argmax(1) == labels).mean():.3f}%   test {100 * (Y_test.argmax(1) == labels_test).mean():.3f}%")

In [ ]:
PCA_K = 100

Xs, Xs_test = X / 255.0, X_test / 255.0
mu = Xs.mean(axis=0, keepdims=True)
_, _, Vh = np.linalg.svd(Xs - mu, full_matrices=False)
ii, jj = np.triu_indices(PCA_K)


def features(Xc):
    """[z, z_i z_j for i <= j] with z = Xc @ Vh[:PCA_K].T."""
    z = Xc @ Vh[:PCA_K].T
    return np.concatenate([z, z[:, ii] * z[:, jj]], axis=1)


def with_bias(P):
    return np.concatenate([P, np.ones((len(P), 1))], axis=1)


A_tr = with_bias(features(Xs - mu))
A_te = with_bias(features(Xs_test - mu))
print(f"coordinates = {A_tr.shape[1]}  (= {PCA_K} linear + {PCA_K * (PCA_K + 1) // 2} products + bias)")

W, _, rank, _ = np.linalg.lstsq(A_tr, Y, rcond=None)
Y_hat, Y_hat_test = A_tr @ W, A_te @ W
print(f"lstsq rank {rank} / {A_tr.shape[1]}   W {W.shape}")

e_tr, e_te = np.abs(Y_hat - Y), np.abs(Y_hat_test - Y_test)
print(f"train RMSE={np.sqrt(np.mean(e_tr**2)):.4e}  max|Δ|={e_tr.max():.4e}")
print(f"test  RMSE={np.sqrt(np.mean(e_te**2)):.4e}  max|Δ|={e_te.max():.4e}")
print(f"train acc {100 * (Y_hat.argmax(1) == labels).mean():.3f}%  vs LeNet {100 * (Y_hat.argmax(1) == Y.argmax(1)).mean():.3f}%")
print(f"test  acc {100 * (Y_hat_test.argmax(1) == labels_test).mean():.3f}%  vs LeNet {100 * (Y_hat_test.argmax(1) == Y_test.argmax(1)).mean():.3f}%")

## Cell 4 — the fitted map as a single function $x\mapsto\hat Y$

Everything below is fixed after the fit: the mean $\mu\in\mathbb{R}^{784}$, the
principal directions $V\in\mathbb{R}^{100\times784}$ (rows of `Vh[:100]`), and the
readout $W\in\mathbb{R}^{5151\times10}$.

Given one image $x\in[0,255]^{28\times28}$:

$$
s=\frac{\mathrm{vec}(x)}{255}\in\mathbb{R}^{784},
\qquad
z=V\,(s-\mu)\in\mathbb{R}^{100}.
$$

The 5151 coordinates are the scores, their upper-triangular products, and a bias:

$$
\phi(x)=\bigl[\,\underbrace{z_1,\dots,z_{100}}_{100},\;
\underbrace{z_iz_j\ (1\le i\le j\le100)}_{5050},\;
\underbrace{1}_{1}\,\bigr]^{\top}\in\mathbb{R}^{5151},
$$

$$
\hat Y(x)=W^{\top}\phi(x)\in\mathbb{R}^{10}.
$$

Because $i\le j$ includes the squares, the product block is exactly a quadratic
form. Writing $a_c$ for the linear weights, $b_c$ for the bias and $Q_c$ for the
symmetric $100\times100$ matrix of logit $c$,

$$
\hat Y_c(x)=b_c+a_c^{\top}z+z^{\top}Q_cz,
\qquad
(Q_c)_{ij}=(Q_c)_{ji}=\tfrac12 W_{\,\pi(i,j),\,c}\ (i\neq j),
\quad (Q_c)_{ii}=W_{\,\pi(i,i),\,c},
$$

where $\pi(i,j)$ is the row of $W$ holding the coefficient of $z_iz_j$. So the
model is one affine plus one quadratic form per logit — a degree-2 polynomial in
the 100 PCA coordinates.

In [ ]:
def compute_Y(x_image):
    """One image -> 10 LeNet-approximating logits, using only mu, Vh, W from the fit.

    x_image: (28, 28), (1, 28, 28) or (784,) with raw pixel values in [0, 255].
    """
    s = np.asarray(x_image, dtype=np.float64).reshape(-1) / 255.0
    z = Vh[:PCA_K] @ (s - mu[0])              # 100 PCA scores
    phi = np.concatenate([z, z[ii] * z[jj], [1.0]])   # 5151 coordinates
    return phi @ W                            # 10 logits


print("phi size", PCA_K + PCA_K * (PCA_K + 1) // 2 + 1)

y0 = compute_Y(X_img[0])
print("compute_Y(X_img[0])   ", np.array2string(y0, precision=4, suppress_small=True))
print("batch row Y_hat[0]    ", np.array2string(Y_hat[0], precision=4, suppress_small=True))
print("LeNet Y[0]            ", np.array2string(Y[0], precision=4, suppress_small=True))

# agreement with the vectorised path (same formula, different evaluation order)
rows = np.arange(0, len(X_img_test), 500)
per_image = np.stack([compute_Y(X_img_test[i]) for i in rows])
print(f"\nmax |compute_Y - A_te@W| over {len(rows)} test images: {np.abs(per_image - Y_hat_test[rows]).max():.3e}")

err = np.abs(per_image - Y_test[rows])
print(f"vs LeNet on those images: max|Δ|={err.max():.4f}  RMSE={np.sqrt(np.mean(err**2)):.4f}")
print(f"argmax agrees with LeNet on {100 * np.mean(per_image.argmax(1) == Y_test[rows].argmax(1)):.1f}% of them")


## Cell 5 — diagonalize $z^{\top}Q_cz$

Each logit is already an affine-plus-quadratic polynomial in the PCA scores:

$$
\hat Y_c(x)=b_c+a_c^{\top}z+z^{\top}Q_cz,
\qquad Q_c=Q_c^{\top}\in\mathbb{R}^{100\times100}.
$$

A real symmetric matrix has an orthonormal eigenbasis $U_c$ and a real diagonal
$\Lambda_c=\mathrm{diag}(\lambda_{c,1},\dots,\lambda_{c,100})$:

$$
Q_c=U_c\Lambda_c U_c^{\top},
\qquad
U_c^{\top}U_c=I.
$$

Change coordinates to $t_c=U_c^{\top}z$. Then the cross terms disappear:

$$
z^{\top}Q_cz=t_c^{\top}\Lambda_ct_c=\sum_{k=1}^{100}\lambda_{c,k}\,t_{c,k}^{2},
\qquad
a_c^{\top}z=\tilde a_c^{\top}t_c,
\quad \tilde a_c=U_c^{\top}a_c,
$$

so

$$
\hat Y_c(x)=b_c+\sum_{k=1}^{100}\Bigl(\tilde a_{c,k}\,t_{c,k}+\lambda_{c,k}\,t_{c,k}^{2}\Bigr).
$$

Cell 5 implements this sum. $U_c$ is **per logit**, so the frames $t_c$ differ
across the 10 outputs. This is a change of basis, not a truncation.

In [ ]:
# Cell 5 — implement  Y_c = b_c + sum_k (a_tilde_{c,k} t_{c,k} + lambda_{c,k} t_{c,k}^2)
compute_Y_products = compute_Y  # cell-4 product form, for a numerical check

a_lin = W[:PCA_K]
b_bias = W[-1]
quad = W[PCA_K:-1]

Q = np.zeros((10, PCA_K, PCA_K), dtype=np.float64)
for t, (i, j) in enumerate(zip(ii, jj)):
    if i == j:
        Q[:, i, j] = quad[t]
    else:
        Q[:, i, j] = Q[:, j, i] = 0.5 * quad[t]

lam = np.empty((10, PCA_K), dtype=np.float64)
U = np.empty((10, PCA_K, PCA_K), dtype=np.float64)
a_tilde = np.empty_like(a_lin)
for c in range(10):
    lam[c], U[c] = np.linalg.eigh(Q[c])
    a_tilde[:, c] = U[c].T @ a_lin[:, c]


def compute_Y(x_image):
    """Ŷ_c = b_c + Σ_k (ã_{c,k} t_{c,k} + λ_{c,k} t_{c,k}²),  t_c = U_cᵀ z."""
    s = np.asarray(x_image, dtype=np.float64).reshape(-1) / 255.0
    z = Vh[:PCA_K] @ (s - mu[0])
    y = np.empty(10, dtype=np.float64)
    for c in range(10):
        t = U[c].T @ z
        y[c] = b_bias[c] + np.sum(a_tilde[:, c] * t + lam[c] * t * t)
    return y


y0 = compute_Y(X_img[0])
print("compute_Y(X_img[0])", np.array2string(y0, precision=4, suppress_small=True))
print(f"vs product form on image 0: {np.abs(y0 - compute_Y_products(X_img[0])).max():.3e}")
rows = np.arange(0, len(X_img_test), 500)
err = np.abs(
    np.stack([compute_Y(X_img_test[i]) for i in rows])
    - np.stack([compute_Y_products(X_img_test[i]) for i in rows])
)
print(f"vs product form over {len(rows)} test images: {err.max():.3e}")


## Cell 6 — operation counts for one image

Counting **multiply**, **add**, and **max of two scalars**. A convolution with
$K$ taps and a bias is $K$ multiplies and $K$ adds per output ($K-1$ to sum
the products, $+1$ for the bias). $\mathrm{ReLU}(u)=\max(u,0)$ is one
max-of-two. A $2\times2$ max-pool window is three max-of-two
($\max(\max(a,b),\max(c,d))$). No fused “MAC” discount.

**LeNet** (this ONNX net, one $28\times28$ image):

$$
\begin{aligned}
\mathrm{conv1}:&\ 28\cdot28\cdot32\cdot(1\cdot5\cdot5)
  &&=627\,200\ \mathrm{mul/add},
  && \mathrm{ReLU}:\ 25\,088\ \max_2,\\
\mathrm{pool1}:&\ 32\cdot14\cdot14\cdot 3
  &&=18\,816\ \max_2,\\
\mathrm{conv2}:&\ 14\cdot14\cdot64\cdot(32\cdot5\cdot5)
  &&=10\,035\,200\ \mathrm{mul/add},
  && \mathrm{ReLU}:\ 12\,544\ \max_2,\\
\mathrm{pool2}:&\ 64\cdot7\cdot7\cdot 3
  &&=9\,408\ \max_2,\\
\mathrm{conv3}:&\ 1024\cdot(64\cdot7\cdot7)
  &&=3\,211\,264\ \mathrm{mul/add},
  && \mathrm{ReLU}:\ 1\,024\ \max_2,\\
\mathrm{conv4}:&\ 10\cdot1024
  &&=10\,240\ \mathrm{mul/add}.
\end{aligned}
$$

**Product readout** (`compute_Y` in cell 4), $V\in\mathbb{R}^{100\times784}$:

$$
\begin{aligned}
s&=x/255 && 784\ \mathrm{mul},\\
z&=V(s-\mu) && 100\cdot784\ \mathrm{mul},\ 100\cdot783\ \mathrm{add},\\
\{z_iz_j\}_{i\le j} && 5050\ \mathrm{mul},\\
\hat Y&=W^{\top}\phi && 10\cdot5151\ \mathrm{mul},\ 10\cdot5150\ \mathrm{add}.
\end{aligned}
$$

**Diagonal readout** (cell 5) replaces the products and $W$ by, for each logit $c$,
$t_c=U_c^{\top}z$ and $\sum_k(\tilde a_{c,k}t_{c,k}+\lambda_{c,k}t_{c,k}^{2})$.
That is $10\cdot100\cdot100$ extra multiplies for the ten eigen-rotations, so
it is **more** arithmetic than the product form, not less.

In [ ]:
def _conv(hout, wout, cout, cin, k):
    K = cin * k * k
    n = hout * wout * cout
    return n * K, n * K  # mul, add (K products + bias)

def _pool_max2(c, h, w):
    return 3 * c * (h // 2) * (w // 2)

lenet = {
    "conv1 32×5×5 on 28×28": (*_conv(28, 28, 32, 1, 5), 0),
    "ReLU after conv1": (0, 0, 28 * 28 * 32),
    "maxpool 2×2 → 14×14": (0, 0, _pool_max2(32, 28, 28)),
    "conv2 64×5×5 on 14×14": (*_conv(14, 14, 64, 32, 5), 0),
    "ReLU after conv2": (0, 0, 14 * 14 * 64),
    "maxpool 2×2 → 7×7": (0, 0, _pool_max2(64, 14, 14)),
    "conv3 1024×7×7 → 1×1": (*_conv(1, 1, 1024, 64, 7), 0),
    "ReLU after conv3": (0, 0, 1024),
    "conv4 10×1×1": (*_conv(1, 1, 10, 1024, 1), 0),
}

prod = {
    "x / 255": (784, 0, 0),
    "s − μ": (0, 784, 0),
    "z = V (s−μ)   V: 100×784": (100 * 784, 100 * 783, 0),
    "z_i z_j  (i ≤ j)": (5050, 0, 0),
    "Wᵀ φ   W: 5151×10": (10 * 5151, 10 * 5150, 0),
}

# per logit: t = Uᵀz → 100×100 mul, 100×99 add
# then 100×(ã·t, t², λ·t²) = 300 mul, and (ãt+λt²) + sum + bias = 200 add
diag = {
    "x / 255": (784, 0, 0),
    "s − μ": (0, 784, 0),
    "z = V (s−μ)": (100 * 784, 100 * 783, 0),
    "t_c = U_cᵀ z  (10 logits)": (10 * 100 * 100, 10 * 100 * 99, 0),
    "Σ (ã t + λ t²) + b": (10 * 300, 10 * 200, 0),
}

def _tot(d):
    return tuple(sum(v[i] for v in d.values()) for i in range(3))

def _show(title, d):
    print(f"\n{title}")
    print(f"{'step':<36} {'mul':>12} {'add':>12} {'max2':>10}")
    for name, (m, a, x) in d.items():
        print(f"{name:<36} {m:12,d} {a:12,d} {x:10,d}")
    m, a, x = _tot(d)
    print(f"{'TOTAL':<36} {m:12,d} {a:12,d} {x:10,d}   arith={m+a+x:,d}")
    return m, a, x

Lm, La, Lx = _show("LeNet  (one 28×28 image, [0,255])", lenet)
Pm, Pa, Px = _show("PCA-100 quadratic, product form (cell 4)", prod)
Dm, Da, Dx = _show("PCA-100 quadratic, diagonal form (cell 5)", diag)

print("\nratios  (LeNet / model)")
print(f"  product form   mul {Lm/Pm:.1f}×   add {La/Pa:.1f}×   all-ops {(Lm+La+Lx)/(Pm+Pa):.1f}×")
print(f"  diagonal form  mul {Lm/Dm:.1f}×   add {La/Da:.1f}×   all-ops {(Lm+La+Lx)/(Dm+Da):.1f}×")
print(f"\nLeNet uses {Lx:,} max(a,b); the quadratic maps use 0.")


## Cell 7 — parameter counts

A **parameter** here is a floating-point number stored for inference (weights and
biases). ONNX reshape-shape integers are omitted.

**LeNet** (this `lenet.onnx`):

$$
\begin{aligned}
\mathrm{conv1}:&\ 32\cdot 1\cdot 5\cdot 5+32 &&=832,\\
\mathrm{conv2}:&\ 64\cdot 32\cdot 5\cdot 5+64 &&=51\,264,\\
\mathrm{conv3}:&\ 1024\cdot 64\cdot 7\cdot 7+1024 &&=3\,212\,288,\\
\mathrm{conv4}:&\ 10\cdot 1024\cdot 1\cdot 1+10 &&=10\,250.
\end{aligned}
$$

Almost all of the mass is `conv3`.

**PCA-100 quadratic (product form).** Inference needs the train mean $\mu$, the
$100$ principal directions $V$, and the readout $W$:

$$
\mu\in\mathbb{R}^{784},\quad
V\in\mathbb{R}^{100\times784},\quad
W\in\mathbb{R}^{5151\times10}.
$$

$W$ is the only block fitted to LeNet logits $Y$. $V$ and $\mu$ come from PCA on
$X/255$. The diagonal form $Q_c=U_c\Lambda_c U_c^{\top}$ is the **same**
$5151\times10$ numbers, just factored; storing dense $U_c$ ($100\times100$ per
logit) uses more memory without adding independent parameters.

In [ ]:
lenet_params = {
    "conv1  32×1×5×5 + bias": 32 * 1 * 5 * 5 + 32,
    "conv2  64×32×5×5 + bias": 64 * 32 * 5 * 5 + 64,
    "conv3  1024×64×7×7 + bias": 1024 * 64 * 7 * 7 + 1024,
    "conv4  10×1024×1×1 + bias": 10 * 1024 * 1 * 1 + 10,
}

quad_params = {
    "μ  (pixel mean)": 784,
    "V  (100 PCs, 784-D)": 100 * 784,
    "W  (5151 coords → 10 logits)": 5151 * 10,
}

# Same map; dense eigen-storage (U is orthonormal, so this over-counts degrees of freedom)
diag_store = {
    "μ": 784,
    "V": 100 * 784,
    "U_c  10 × 100×100": 10 * 100 * 100,
    "λ_c  10 × 100": 10 * 100,
    "ã_c  10 × 100": 10 * 100,
    "b     10": 10,
}

def _show(title, d):
    print(title)
    print(f"{'block':<40} {'count':>12}")
    for k, v in d.items():
        print(f"{k:<40} {v:12,d}")
    t = sum(d.values())
    print(f"{'TOTAL':<40} {t:12,d}")
    print()
    return t

Lt = _show("LeNet  (learned weights + biases)", lenet_params)
Qt = _show("PCA-100 quadratic, stored for inference", quad_params)
Dt = _show("Same map, dense diagonal storage (not extra DOF)", diag_store)

print(f"readout W only:                 {quad_params['W  (5151 coords → 10 logits)']:,}")
print(f"independent polynomial coeffs:  {5151 * 10:,}   (= W; Q_c is 5050 unique entries × 10 + linear + bias)")
print(f"LeNet / (μ+V+W) = {Lt / Qt:.1f}×")
print(f"LeNet / W only  = {Lt / (5151 * 10):.1f}×")
print(f"conv3 share of LeNet: {100 * lenet_params['conv3  1024×64×7×7 + bias'] / Lt:.2f}%")


## Cell 8 — conclusion (what is safe to claim)

This notebook fits a **degree-2 polynomial in 100 PCA scores** of $X/255$ to this
ONNX LeNet's 10 logits, then evaluates it with `compute_Y`. It is **not** a
lossless rewrite of LeNet. Counts below are for **one** $28\times28$ image
(cell 6) and for the **full stored model** (cell 7): every float needed at
inference, no subsets.

**Accuracy (measured, MNIST).**

- LeNet: $97.95\%$ train / $97.37\%$ test.
- This map: $97.06\%$ train / $96.47\%$ test (argmax vs labels).
- Argmax agreement with LeNet: $97.22\%$ train / $96.37\%$ test.
- Pointwise logits: train RMSE $\approx 13.1$, $\max|Y-\hat Y|\approx 122$.
  The $10^{-4}$ target on every coordinate is **not** met.

**Operations (full forward pass, product form vs LeNet).**

| | mul | add | $\max(a,b)$ |
|---|---:|---:|---:|
| LeNet | $13\,883\,904$ | $13\,883\,904$ | $66\,880$ |
| full quadratic, $\phi$-readout (cell 4) | $135\,744$ | $130\,584$ | $0$ |
| same map, diagonal $Q_c$ (cell 5) | $182\,184$ | $180\,084$ | $0$ |

Safe claim: the **product** implementation of the full map uses on the order of
**$100\times$ fewer** multiplies and adds than this LeNet ($\approx 102\times$
mul, $\approx 104\times$ mul+add), and **no** two-element max. That is a
**factor of $\sim 100$**, not $100$ fewer operations.

**Not** a safe claim for cell 5: the same full model in the eigenbasis
**increases** arithmetic (ten $100\times100$ rotations) to about **$76\times$**
fewer muls than LeNet.

**Full model parameters** (every float `compute_Y` stores: $\mu$, $V$, and $W$;
for LeNet, every conv weight and bias).

| full model | floats |
|---|---:|
| LeNet | $3\,274\,634$ |
| quadratic ($\mu+V+W$) | $130\,694$ |

$$
\frac{3\,274\,634}{130\,694}\approx 25.1.
$$

Safe claim: the **full** quadratic model uses about **$25\times$ fewer** stored
floats than the **full** LeNet. Ratios that drop $V$ or $\mu$, or that expand
$Q_c$ into dense $U_c$, are not full-model comparisons and are not used here.

**One-sentence version.** This is a cheaper **approximation** of LeNet logits
(about **$1$ point** of test accuracy, large logit error): **$\sim 100\times$**
fewer add/mul ops in the product form, **$\sim 25\times$** fewer full-model
parameters, and **not** an exact or $10^{-4}$-faithful substitute.